In [32]:

from datasets import load_dataset


#loading from hugging face
dataset = load_dataset("tattabio/ec_classification")
## making test and train set
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
##sanity check to make sure dims are the same as on the DGEB paper dimensions
print(train_df.shape)
print(test_df.shape)

(512, 3)
(128, 3)


In [33]:
train_df.head()
## checking what the label and seq loomks like

,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


In [34]:
x_train_seq = train_df["Sequence"].tolist()
y_train_labels = train_df["Label"].tolist()

x_test_seq = test_df["Sequence"].tolist()
y_test_labels = test_df["Label"].tolist()

from sklearn.ensemble import RandomForestClassifier
## defining model
model100 = RandomForestClassifier(n_estimators=100, random_state=42,)
model200 = RandomForestClassifier(n_estimators=200, random_state=42,)


In [35]:

import dgeb
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
## first input which is just the embeddings i am using dgeb embedding model to make each seq into a vector
embedding_model = dgeb.get_model("facebook/esm2_t6_8M_UR50D")



Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [36]:
## converting train data to numerical vectors
x_train_input1 = embedding_model.encode(x_train_seq)[:, -1, :]

/opt/anaconda3/envs/knnlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
encoding:   0%|          | 0/4 [00:00<?, ?it/s]/opt/anaconda3/envs/knnlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
python(29919) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29931) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(29933) 

In [37]:
## doing same for test data
## broke into two cells so we would avoid rerunning

import numpy as np
x_test_p1 = embedding_model.encode(x_test_seq[:127])[:, -1, :]
last_test = embedding_model.encode(x_test_seq[127:])[:, -1, :]
x_test_input1 = np.vstack([x_test_p1, last_test])


python(30235) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30239) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30247) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30248) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30256) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30258) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30259) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30268) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30273) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30274) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(30282) Malloc

In [38]:
from sklearn.metrics import accuracy_score, f1_score 
## checking shape
print(x_train_input1.shape)
print(x_test_input1.shape)
## now fitting model on this info
model100.fit(x_train_input1, y_train_labels)
predictions_input1 = model100.predict(x_test_input1)

## accuracy and f1 score for input 1
print("accuracy for model with 100 iters:", accuracy_score(y_test_labels, predictions_input1))
print("f1 for model with 100 iters:", f1_score(y_test_labels, predictions_input1, average="macro"))

## doing it for the model with 200 trees
model200.fit(x_train_input1, y_train_labels)
predictions_input1 = model200.predict(x_test_input1)

## accuracy and f1 score for input 1
print("accuracy  for model with 200 iters:", accuracy_score(y_test_labels, predictions_input1))
print("f1  for model with 200 iters:", f1_score(y_test_labels, predictions_input1, average="macro"))

(512, 320)
(128, 320)
accuracy for model with 100 iters: 0.4296875
f1 for model with 100 iters: 0.37473958333333335
accuracy  for model with 200 iters: 0.4296875
f1  for model with 200 iters: 0.3794270833333333


In [41]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score 
## now doing input 2 which is k mer counts of 2 and 3

def kmerFunction(seq, k):
    return " ".join([seq[i:i+k] for i in range(len(seq) - k + 1)])


train_kmer2 = [kmerFunction(seq, 2) for seq in x_train_seq]
test_kmer2 = [kmerFunction(seq, 2) for seq in x_test_seq]

train_kmer3 = [kmerFunction(seq, 3) for seq in x_train_seq]
test_kmer3 = [kmerFunction(seq, 3) for seq in x_test_seq]

## for kmer 2
helper = CountVectorizer()
x_train_input2_kmer2 = helper.fit_transform(train_kmer2)
x_test_input2_kmer2 = helper.transform(test_kmer2)
## model for 100 trees
model2kmer2_100iters = RandomForestClassifier(n_estimators=100, random_state=42,)
model2kmer2_100iters.fit(x_train_input2_kmer2, y_train_labels)
predictions_input2kmer2 = model2kmer2_100iters.predict(x_test_input2_kmer2)
## printing accuracy and f1 score
print("accuracy for kmer2 with 100 trees:", accuracy_score(y_test_labels, predictions_input2kmer2))
print("f1 for kmer2 with 100 trees:", f1_score(y_test_labels, predictions_input2kmer2, average="macro"))
## model for 200 trees
model2kmer2_200iters = RandomForestClassifier(n_estimators=200, random_state=42,)
model2kmer2_200iters.fit(x_train_input2_kmer2, y_train_labels)
predictions_input2kmer2 = model2kmer2_200iters.predict(x_test_input2_kmer2)
## printing accuracy and f1 score
print("accuracy for kmer2 with 200 trees:", accuracy_score(y_test_labels, predictions_input2kmer2))
print("f1 for kmer2 with 200 trees:", f1_score(y_test_labels, predictions_input2kmer2, average="macro"))

## for kmer 3
helper = CountVectorizer()
x_train_input2_kmer3 = helper.fit_transform(train_kmer3)
x_test_input2_kmer3 = helper.transform(test_kmer3)
## model for 100 trees
model2kmer3_100iters = RandomForestClassifier(n_estimators=100, random_state=42,)
model2kmer3_100iters.fit(x_train_input2_kmer3, y_train_labels)
predictions_input2kmer3 = model2kmer3_100iters.predict(x_test_input2_kmer3)
## printing accuracy and f1 score
print("accuracy for kmer3 with 100 trees:", accuracy_score(y_test_labels, predictions_input2kmer3))
print("f1 for kmer3 with 100 trees:", f1_score(y_test_labels, predictions_input2kmer3, average="macro"))
## model for 200 trees
model2kmer3_200iters = RandomForestClassifier(n_estimators=200, random_state=42,)
model2kmer3_200iters.fit(x_train_input2_kmer3, y_train_labels)
predictions_input2kmer3 = model2kmer3_200iters.predict(x_test_input2_kmer3)
## printing accuracy and f1 score
print("accuracy for kmer3 with 200 trees:", accuracy_score(y_test_labels, predictions_input2kmer3))
print("f1 for kmer3 with 200 trees:", f1_score(y_test_labels, predictions_input2kmer3, average="macro"))

accuracy for kmer2 with 100 trees: 0.1015625
f1 for kmer2 with 100 trees: 0.06979166666666667
accuracy for kmer2 with 200 trees: 0.125
f1 for kmer2 with 200 trees: 0.08530505952380951
accuracy for kmer3 with 100 trees: 0.1171875
f1 for kmer3 with 100 trees: 0.08645833333333333
accuracy for kmer3 with 200 trees: 0.109375
f1 for kmer3 with 200 trees: 0.07135416666666666


In [47]:
## now doing input 3 which is weighted kmer
from sklearn.feature_extraction.text import TfidfVectorizer

## defining the weighted helper
weighted2 =  TfidfVectorizer()
x_train_input3_kmer2 = weighted2.fit_transform(train_kmer2)
x_test_input3_kmer2 = weighted2.transform(test_kmer2)

weighted3 =  TfidfVectorizer()
x_train_input3_kmer3 = weighted3.fit_transform(train_kmer3)
x_test_input3_kmer3 = weighted3.transform(test_kmer3)


## making model3 for kmer 2 
model3_kmer2_100trees = RandomForestClassifier(n_estimators=100, random_state=42,)
model3_kmer2_100trees.fit(x_train_input3_kmer2, y_train_labels)
predictions_input3_kmer2_100trees = model3_kmer2_100trees.predict(x_test_input3_kmer2)
## f1 and accuracy scores
print("accuracy for kmer 2 and 100 trees:", accuracy_score(y_test_labels, predictions_input3_kmer2_100trees))
print("f1 for kmer 2 and 100 trees:", f1_score(y_test_labels, predictions_input3_kmer2_100trees, average="macro"))

model3_kmer2_200trees = RandomForestClassifier(n_estimators=200, random_state=42,)
model3_kmer2_200trees.fit(x_train_input3_kmer2, y_train_labels)
predictions_input3_kmer2_200trees = model3_kmer2_200trees.predict(x_test_input3_kmer2)
## f1 and accuracy scores
print("accuracy for kmer 2 and 200 trees:", accuracy_score(y_test_labels, predictions_input3_kmer2_200trees))
print("f1 for kmer 2 and 200 trees:", f1_score(y_test_labels, predictions_input3_kmer2_200trees, average="macro"))

## making model3 for kmer 3 
model3_kmer3_100trees = RandomForestClassifier(n_estimators=100, random_state=42,)
model3_kmer3_100trees.fit(x_train_input3_kmer3, y_train_labels)
predictions_input3_kmer3_100trees = model3_kmer3_100trees.predict(x_test_input3_kmer3)
## f1 and accuracy scores
print("accuracy for kmer 3 and 100 trees:", accuracy_score(y_test_labels, predictions_input3_kmer3_100trees))
print("f1 for kmer 3 and 100 trees:", f1_score(y_test_labels, predictions_input3_kmer3_100trees, average="macro"))

model3_kmer3_200trees = RandomForestClassifier(n_estimators=200, random_state=42,)
model3_kmer3_200trees.fit(x_train_input3_kmer3, y_train_labels)
predictions_input3_kmer3_200trees = model3_kmer3_200trees.predict(x_test_input3_kmer3)
## f1 and accuracy scores
print("accuracy for kmer 3 and 200 trees:", accuracy_score(y_test_labels, predictions_input3_kmer3_200trees))
print("f1 for kmer 3 and 200 trees:", f1_score(y_test_labels, predictions_input3_kmer3_200trees, average="macro"))

accuracy for kmer 2 and 100 trees: 0.1171875
f1 for kmer 2 and 100 trees: 0.07569444444444444
accuracy for kmer 2 and 200 trees: 0.140625
f1 for kmer 2 and 200 trees: 0.10546875
accuracy for kmer 3 and 100 trees: 0.1640625
f1 for kmer 3 and 100 trees: 0.12161458333333333
accuracy for kmer 3 and 200 trees: 0.1484375
f1 for kmer 3 and 200 trees: 0.11588541666666666
